In [1]:
##movielens ratings dataset

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName('rec').getOrCreate()

24/07/13 02:57:22 WARN Utils: Your hostname, javad resolves to a loopback address: 127.0.1.1; using 192.168.1.39 instead (on interface wlo1)
24/07/13 02:57:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/13 02:57:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

In [5]:
data = spark.read.csv('movielens_ratings.csv',inferSchema=True,header=True)

In [6]:
data.head()

Row(movieId=2, rating=3.0, userId=0)

In [7]:
data.describe().show()

+-------+------------------+------------------+------------------+
|summary|           movieId|            rating|            userId|
+-------+------------------+------------------+------------------+
|  count|              1501|              1501|              1501|
|   mean| 49.40572951365756|1.7741505662891406|14.383744170552964|
| stddev|28.937034065088994| 1.187276166124803| 8.591040424293272|
|    min|                 0|               1.0|                 0|
|    max|                99|               5.0|                29|
+-------+------------------+------------------+------------------+



In [8]:
# Smaller dataset so we will use 0.8 / 0.2
(training, test) = data.randomSplit([0.8, 0.2])

In [9]:
# Build the recommendation model using ALS on the training data
als = ALS(maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId", ratingCol="rating")
model = als.fit(training)

24/07/13 02:58:33 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/07/13 02:58:33 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [10]:
# Evaluate the model by computing the RMSE on the test data
predictions = model.transform(test)

In [11]:
predictions.show()

+-------+------+------+-----------+
|movieId|rating|userId| prediction|
+-------+------+------+-----------+
|      2|   1.0|    12|  1.5220678|
|      3|   1.0|     1|  1.1047972|
|      3|   1.0|    13| 0.82413405|
|      5|   1.0|    13|  1.2572184|
|      5|   1.0|     6|  1.3900186|
|      5|   3.0|    16| 0.17572865|
|      1|   1.0|     3| -1.1398114|
|      4|   1.0|     5|  2.0459409|
|      1|   4.0|    15|  1.6230327|
|      2|   1.0|    17|-0.41877165|
|      2|   3.0|     9|  2.4333115|
|      2|   4.0|     8|  3.2198303|
|      2|   1.0|    23|   4.493871|
|      4|   1.0|    23|  1.9897691|
|      1|   1.0|     7| 0.17277952|
|      0|   3.0|    10|  1.4936402|
|      1|   1.0|    14|  1.7200608|
|      3|   3.0|    14|  1.1867161|
|      2|   3.0|     0|  3.0769405|
|      3|   1.0|     0|  0.4836671|
+-------+------+------+-----------+
only showing top 20 rows



In [12]:
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

Root-mean-square error = 1.619724839918985


In [13]:
single_user = test.filter(test['userId']==11).select(['movieId','userId'])

In [14]:
# User had 10 ratings in the test data set 
# Realistically this should be some sort of hold out set!
single_user.show()

+-------+------+
|movieId|userId|
+-------+------+
|     11|    11|
|     16|    11|
|     18|    11|
|     21|    11|
|     30|    11|
|     36|    11|
|     37|    11|
|     38|    11|
|     64|    11|
|     66|    11|
|     70|    11|
|     71|    11|
|     75|    11|
|     76|    11|
|     81|    11|
|     82|    11|
|     86|    11|
+-------+------+



In [15]:
reccomendations = model.transform(single_user)

In [16]:
reccomendations.orderBy('prediction',ascending=False).show()

+-------+------+------------+
|movieId|userId|  prediction|
+-------+------+------------+
|     64|    11|   3.8331394|
|     66|    11|    3.414401|
|     38|    11|   3.0980864|
|     21|    11|   2.2827716|
|     16|    11|   2.0325763|
|     76|    11|   1.6235011|
|     11|    11|   1.6126218|
|     82|    11|   1.2038786|
|     81|    11|   1.1207936|
|     86|    11|   1.0164025|
|     37|    11|  0.95407665|
|     18|    11|   0.7079979|
|     30|    11|  0.67532253|
|     36|    11| 0.107280105|
|     70|    11|-0.059333205|
|     75|    11| -0.14515796|
|     71|    11|  -1.0319958|
+-------+------+------------+



In [17]:
## meal info dataset

In [23]:
import pandas as pd

In [24]:
df = pd.read_csv('Meal_Info.csv')

In [25]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
movieId,1501.0,49.405730,28.937034,0.0,24.0,50.0,74.0,99.0
rating,1501.0,1.774151,1.187276,1.0,1.0,1.0,2.0,5.0
userId,1501.0,14.383744,8.591040,0.0,7.0,14.0,22.0,29.0
mealskew,486.0,15.502058,9.250634,0.0,7.0,15.0,23.0,31.0


In [27]:
df.corr(numeric_only=True)

,movieId,rating,userId,mealskew
movieId,1.000000,0.036569,0.003267,1.000000
rating,0.036569,1.000000,0.056411,0.131044
userId,0.003267,0.056411,1.000000,0.017888
mealskew,1.000000,0.131044,0.017888,1.000000


In [28]:
import numpy as np
df['mealskew'] = df['movieId'].apply(lambda id: np.nan if id > 31 else id)

In [29]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
movieId,1501.0,49.405730,28.937034,0.0,24.0,50.0,74.0,99.0
rating,1501.0,1.774151,1.187276,1.0,1.0,1.0,2.0,5.0
userId,1501.0,14.383744,8.591040,0.0,7.0,14.0,22.0,29.0
mealskew,486.0,15.502058,9.250634,0.0,7.0,15.0,23.0,31.0


In [30]:
mealmap = { 2. : "Chicken Curry",   
           3. : "Spicy Chicken Nuggest",   
           5. : "Hamburger",   
           9. : "Taco Surprise",  
           11. : "Meatloaf",  
           12. : "Ceaser Salad",  
           15. : "BBQ Ribs",  
           17. : "Sushi Plate",  
           19. : "Cheesesteak Sandwhich",  
           21. : "Lasagna",  
           23. : "Orange Chicken",
           26. : "Spicy Beef Plate",  
           27. : "Salmon with Mashed Potatoes",  
           28. : "Penne Tomatoe Pasta",  
           29. : "Pork Sliders",  
           30. : "Vietnamese Sandwich",  
           31. : "Chicken Wrap",  
           np.nan: "Cowboy Burger",   
           4. : "Pretzels and Cheese Plate",   
           6. : "Spicy Pork Sliders",  
           13. : "Mandarin Chicken PLate",  
           14. : "Kung Pao Chicken",
           16. : "Fried Rice Plate",  
           8. : "Chicken Chow Mein",  
           10. : "Roasted Eggplant ",  
           18. : "Pepperoni Pizza",  
           22. : "Pulled Pork Plate",   
           0. : "Cheese Pizza",   
           1. : "Burrito",   
           7. : "Nachos",  
           24. : "Chili",  
           20. : "Southwest Salad",  
           25.: "Roast Beef Sandwich"}

In [31]:
df['meal_name'] = df['mealskew'].map(mealmap)

In [32]:
df.to_csv('Meal_Info.csv',index=False)

In [33]:
from pyspark.sql import SparkSession

In [34]:
spark = SparkSession.builder.appName('recconsulting').getOrCreate()

24/07/13 03:06:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [35]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

In [36]:
data = spark.read.csv('Meal_Info.csv',inferSchema=True,header=True)

In [37]:
(training, test) = data.randomSplit([0.8, 0.2])

In [39]:
# Evaluate the model by computing the RMSE on the test data
predictions = model.transform(test)

predictions.show()

evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

+-------+------+------+--------+--------------------+----------+
|movieId|rating|userId|mealskew|           meal_name|prediction|
+-------+------+------+--------+--------------------+----------+
|      3|   1.0|    28|     3.0|Spicy Chicken Nug...| 0.9552319|
|      4|   1.0|    12|     4.0|Pretzels and Chee...|0.50536036|
|      4|   2.0|     1|     4.0|Pretzels and Chee...| 2.1306074|
|      4|   2.0|    13|     4.0|Pretzels and Chee...| 1.9216841|
|      6|   1.0|    13|     6.0|  Spicy Pork Sliders|0.98853505|
|      2|   3.0|     6|     2.0|       Chicken Curry| 2.8689017|
|      6|   1.0|     6|     6.0|  Spicy Pork Sliders| 0.6470004|
|      0|   1.0|     5|     0.0|        Cheese Pizza| 1.0020752|
|      4|   1.0|     5|     4.0|Pretzels and Chee...| 2.0459409|
|      1|   1.0|    19|     1.0|             Burrito|  1.028919|
|      3|   2.0|     8|     3.0|Spicy Chicken Nug...| 1.9244133|
|      4|   3.0|    10|     4.0|Pretzels and Chee...| 2.8628676|
|      2|   1.0|    25|  